# 🥛 Dairy Supply Chain Demand Forecasting — Stage 1: ARIMA (All Series)
### Thesis: Hybrid Time Series & ML Approach for Multi-Echelon Supply Chain Demand Forecasting

This notebook runs **ARIMA across every product×location series in all 3 echelons**, then saves the ARIMA residuals + features so the **Random Forest** stage (Notebook 2) can learn the nonlinear part.

**Design decisions (justified by the data):**
- Series are short (E2/E3 median ≈ 8 months), so **non-seasonal ARIMA** is used. Seasonality (Month, Season, festival, Demand_Lag_12) is handed to **Random Forest** in Stage 2 — a clean ARIMA(linear)+RF(nonlinear) split.
- A **minimum-length gate** (≥ 10 observed months) skips series too short to fit+test. Surviving counts are reported (E1 150, E2 104, E3 48).
- Series are **sparse over a wide span** (e.g. ~18 observed months scattered across ~46 calendar months). We therefore model the **ordered observed sequence** rather than reindexing to a continuous monthly axis — reindexing would invent dozens of phantom months. Every residual row keeps its real precomputed features for RF.
- Order is chosen by **AIC search on the TRAIN split only** (no test leakage). No `pmdarima` dependency.
- Split is **chronological** (no shuffling). Metrics are computed in **original units**.

**Pipeline:** Load → per-series loop {split → order search → walk-forward forecast → metrics → residuals} → aggregate metrics per echelon → save residual file for RF.

## Block 1 — Imports

In [ ]:
import os, warnings, itertools, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams.update({'figure.dpi':130,'axes.grid':True,'grid.alpha':.3,'grid.linestyle':'--',
                     'axes.spines.top':False,'axes.spines.right':False})
PALETTE = ['#1B3A6B','#E8722A','#1E7B4A','#6B4C9A','#27847A','#C0392B']

# --- Global config (used by EDA and the modeling loop) ---
MIN_MONTHS = 10     # length gate: skip series with fewer real observed months
TEST_RATIO = 0.2    # chronological hold-out fraction
MIN_TEST   = 2      # at least this many test points
print('Libraries ready. Gate =', MIN_MONTHS, 'months.')

## Block 2 — Paths

In [ ]:
# EDIT THIS to wherever your file lives. On your PC it was C:\Badhan Thesis
BASE_DIR    = r'C:\Badhan Thesis'
DATA_FILE   = os.path.join(BASE_DIR, 'dairy_thesis_dataset_3echelons.xlsx')
FIGURES_DIR = os.path.join(BASE_DIR, 'ARIMA_Figures')
RESULTS_DIR = os.path.join(BASE_DIR, 'ARIMA_Results')
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Fallback: if not found, look in the current folder
if not os.path.exists(DATA_FILE):
    alt = 'dairy_thesis_dataset_3echelons.xlsx'
    if os.path.exists(alt):
        DATA_FILE = alt
        print('Using dataset in current folder.')
print('Dataset:', DATA_FILE, '| exists:', os.path.exists(DATA_FILE))

## Block 3 — Load all 3 echelons

In [ ]:
def load_echelon(sheet):
    df = pd.read_excel(DATA_FILE, sheet_name=sheet, header=1)   # header row 2 (banner row above)
    df['YearMonth'] = pd.to_datetime(df['YearMonth'], format='%Y-%m')
    return df.sort_values('YearMonth').reset_index(drop=True)

e1 = load_echelon('Echelon_1_Farm')
e2 = load_echelon('Echelon_2_Distribution')
e3 = load_echelon('Echelon_3_Retail')
print('E1', e1.shape, '| E2', e2.shape, '| E3', e3.shape)
print('Date range:', e1['YearMonth'].min().date(), '->', e1['YearMonth'].max().date())

## Block 3b — Exploratory Data Analysis (visuals)

Before modeling: total monthly demand per echelon, demand by product, seasonality, festival effect, and the distribution of series lengths (which motivates the length gate).

In [ ]:
# EDA-1: total monthly demand trend per echelon, with high-festival months marked
fig, axes = plt.subplots(3,1, figsize=(13,9))
cfg = [(e1,'Quantity_Produced','E1 Farm — Production',PALETTE[0]),
       (e2,'Quantity_Sold','E2 Distribution — Sales',PALETTE[1]),
       (e3,'Quantity_Sold','E3 Retail — Sales',PALETTE[2])]
for ax,(df,t,title,c) in zip(axes,cfg):
    m = df.groupby('YearMonth')[t].sum()
    ax.plot(m.index, m.values, color=c, lw=2, marker='o', ms=3)
    ax.fill_between(m.index, m.values, alpha=.12, color=c)
    fest = df[df['Is_High_Festival']==1].groupby('YearMonth')[t].sum()
    if len(fest): ax.scatter(fest.index, fest.values, color=PALETTE[5], s=55, marker='*',
                             zorder=5, label='High-festival month')
    ax.set_title(title, fontweight='bold'); ax.set_ylabel('Quantity'); ax.legend(loc='upper left')
fig.suptitle('Total Monthly Demand by Echelon', fontweight='bold', y=1.0)
plt.tight_layout(); plt.savefig(os.path.join(FIGURES_DIR,'eda1_monthly_trend.png'), dpi=150); plt.show()

In [ ]:
# EDA-2: demand by product (E2) + monthly seasonality + festival-level effect
fig, axes = plt.subplots(1,3, figsize=(18,5))
# (a) total demand by product
pm = e2.groupby('Product Name')['Quantity_Sold'].sum().sort_values()
axes[0].barh(pm.index, pm.values, color=PALETTE[1], edgecolor='white')
axes[0].set_title('E2 — Total demand by product', fontweight='bold'); axes[0].set_xlabel('Quantity Sold')
# (b) seasonality by month
mm = e2.groupby('Month')['Quantity_Sold'].mean()
axes[1].plot(mm.index, mm.values, color=PALETTE[0], lw=2, marker='o')
axes[1].set_xticks(range(1,13)); axes[1].set_title('E2 — Mean demand by month', fontweight='bold')
axes[1].set_xlabel('Month'); axes[1].set_ylabel('Mean Quantity Sold')
# (c) festival spike effect
fl = e2.groupby('Festival_Spike_Level')['Quantity_Sold'].mean()
bars = axes[2].bar(fl.index.astype(str), fl.values,
                   color=[PALETTE[3],PALETTE[2],PALETTE[1],PALETTE[5]][:len(fl)], edgecolor='white')
axes[2].set_title('E2 — Mean demand by festival level', fontweight='bold')
axes[2].set_xlabel('Festival Spike Level (0–3)'); axes[2].set_ylabel('Mean Quantity Sold')
plt.tight_layout(); plt.savefig(os.path.join(FIGURES_DIR,'eda2_product_season_festival.png'), dpi=150); plt.show()

In [ ]:
# EDA-3: distribution of series lengths per echelon — this MOTIVATES the >=10-month gate
fig, axes = plt.subplots(1,3, figsize=(16,4))
for ax,(df,k,name,c) in zip(axes,[(e1,['Location','Product Name'],'E1 Farm',PALETTE[0]),
                                  (e2,['Location','Product Name','Sales Channel'],'E2 Dist',PALETTE[1]),
                                  (e3,['Location','Product Name'],'E3 Retail',PALETTE[2])]):
    lens = df.groupby(k)['YearMonth'].nunique()
    ax.hist(lens.values, bins=range(0, int(lens.max())+3), color=c, edgecolor='white', alpha=.85)
    ax.axvline(MIN_MONTHS, color=PALETTE[5], ls='--', lw=2, label=f'gate = {MIN_MONTHS}')
    ax.set_title(f'{name} — series-length distribution', fontweight='bold')
    ax.set_xlabel('Observed months'); ax.set_ylabel('# series'); ax.legend()
fig.suptitle('Series Length Distribution (justifies the minimum-length gate)', fontweight='bold', y=1.03)
plt.tight_layout(); plt.savefig(os.path.join(FIGURES_DIR,'eda3_series_lengths.png'), dpi=150); plt.show()

### Extra EDA — correlations, distributions, seasonality & festival lift

Beyond the trend/seasonality views above, the next cells add: a **correlation heatmap** of the
numeric demand drivers per echelon (justifies the lag/rolling/festival features used by RF),
**demand distributions** (raw + log, exposing the heavy right tail that motivates median metrics),
a normalised **seasonality index** comparing echelons, and a quantified **festival demand lift**.
All figures are written to `FIGURES_DIR`.

In [ ]:
# EDA-4: correlation heatmap of numeric demand drivers (per echelon)
# Shows how lags, rolling stats, festival score and the target move together —
# this justifies which signals are handed to Random Forest in Stage 2.
import numpy as np
def corr_panel(ax, df, target, title, cmap='RdBu_r'):
    cols = [target,'Demand_Lag_1','Demand_Lag_2','Demand_Lag_3','Demand_Lag_12',
            'Rolling_Avg_3M','Rolling_Avg_6M','Rolling_Std_3M',
            'Total_Festival_Score','Region_Festival_Boost','Month','Quarter']
    cols = [c for c in cols if c in df.columns]
    C = df[cols].corr()
    im = ax.imshow(C.values, vmin=-1, vmax=1, cmap=cmap, aspect='auto')
    ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=90, fontsize=7)
    ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols, fontsize=7)
    for i in range(len(cols)):
        for j in range(len(cols)):
            ax.text(j, i, f'{C.values[i,j]:.2f}', ha='center', va='center',
                    fontsize=6, color='black' if abs(C.values[i,j])<0.6 else 'white')
    ax.set_title(title, fontweight='bold', fontsize=10)
    return im

fig, axes = plt.subplots(1,3, figsize=(20,6))
im = corr_panel(axes[0], e1, 'Quantity_Produced', 'E1 Farm — driver correlations')
corr_panel(axes[1], e2, 'Quantity_Sold', 'E2 Distribution — driver correlations')
corr_panel(axes[2], e3, 'Quantity_Sold', 'E3 Retail — driver correlations')
fig.colorbar(im, ax=axes, fraction=0.012, pad=0.02, label='Pearson r')
fig.suptitle('Correlation of demand with lag / rolling / festival drivers', fontweight='bold', y=1.02)
plt.savefig(os.path.join(FIGURES_DIR,'eda4_correlation_heatmaps.png'), dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# EDA-5: target demand distributions (raw + log) per echelon — shows skew/heavy tails
fig, axes = plt.subplots(2,3, figsize=(17,8))
cfg = [(e1,'Quantity_Produced','E1 Farm',PALETTE[0]),
       (e2,'Quantity_Sold','E2 Distribution',PALETTE[1]),
       (e3,'Quantity_Sold','E3 Retail',PALETTE[2])]
for j,(df,t,name,c) in enumerate(cfg):
    v = df[t].dropna(); v = v[v>=0]
    axes[0,j].hist(v.clip(upper=v.quantile(.99)), bins=40, color=c, edgecolor='white', alpha=.85)
    axes[0,j].axvline(v.median(), color=PALETTE[5], ls='--', lw=2, label=f'median={v.median():.0f}')
    axes[0,j].set_title(f'{name} — demand (raw, 99% clip)', fontweight='bold'); axes[0,j].legend(fontsize=8)
    axes[1,j].hist(np.log1p(v), bins=40, color=c, edgecolor='white', alpha=.85)
    axes[1,j].set_title(f'{name} — demand (log1p)', fontweight='bold'); axes[1,j].set_xlabel('log(1+Quantity)')
fig.suptitle('Demand distributions per echelon — raw (top) vs log (bottom)', fontweight='bold', y=1.0)
plt.tight_layout(); plt.savefig(os.path.join(FIGURES_DIR,'eda5_target_distributions.png'), dpi=150); plt.show()

In [ ]:
# EDA-6: seasonality detail — mean demand by Season and by Quarter (all echelons)
fig, axes = plt.subplots(1,2, figsize=(15,5))
season_order = ['Winter','Spring','Summer','Monsoon','Autumn','Post-Monsoon']
for df,t,name,c in cfg:
    if 'Season' in df.columns:
        ss = df.groupby('Season')[t].mean()
        ss = ss.reindex([s for s in season_order if s in ss.index]).dropna()
        if len(ss)==0: ss = df.groupby('Season')[t].mean()
        axes[0].plot(range(len(ss)), ss.values/ss.mean(), marker='o', lw=2, label=name, color=c)
        axes[0].set_xticks(range(len(ss))); axes[0].set_xticklabels(ss.index, rotation=20)
    qq = df.groupby('Quarter')[t].mean()
    axes[1].plot(qq.index, qq.values/qq.mean(), marker='s', lw=2, label=name, color=c)
axes[0].axhline(1, color='gray', ls=':'); axes[0].set_title('Seasonal index (mean demand / overall mean)', fontweight='bold')
axes[0].set_ylabel('Relative demand'); axes[0].legend(fontsize=8)
axes[1].axhline(1, color='gray', ls=':'); axes[1].set_xticks([1,2,3,4])
axes[1].set_title('Quarterly index (mean demand / overall mean)', fontweight='bold')
axes[1].set_xlabel('Quarter'); axes[1].set_ylabel('Relative demand'); axes[1].legend(fontsize=8)
fig.suptitle('Seasonality across echelons (normalised so echelons are comparable)', fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig(os.path.join(FIGURES_DIR,'eda6_seasonality_index.png'), dpi=150); plt.show()

In [ ]:
# EDA-7: festival effect quantified — demand lift in festival vs non-festival months
fig, ax = plt.subplots(figsize=(9,5))
labels, lifts, cols = [], [], []
for df,t,name,c in cfg:
    if 'Is_Festival_Month' in df.columns:
        g = df.groupby('Is_Festival_Month')[t].mean()
        if 0 in g.index and 1 in g.index and g[0] > 0:
            lift = (g[1]-g[0])/g[0]*100
            labels.append(name); lifts.append(lift); cols.append(c)
bars = ax.bar(labels, lifts, color=cols, edgecolor='white')
for b,l in zip(bars, lifts):
    ax.text(b.get_x()+b.get_width()/2, b.get_height(), f'{l:+.1f}%',
            ha='center', va='bottom', fontweight='bold')
ax.axhline(0, color='gray', ls=':')
ax.set_ylabel('Demand lift in festival months (%)')
ax.set_title('Festival demand lift by echelon (festival vs non-festival mean)', fontweight='bold')
plt.tight_layout(); plt.savefig(os.path.join(FIGURES_DIR,'eda7_festival_lift.png'), dpi=150); plt.show()

## Block 4 — Echelon configuration

Each echelon is defined by its target, its grouping keys (what makes one time series), and a label.
- **E1 Farm:** target `Quantity_Produced`, grouped by Location × Product.
- **E2 Distribution:** target `Quantity_Sold`, grouped by Location × Product × Sales Channel.
- **E3 Retail:** target `Quantity_Sold`, grouped by Location × Product (summed across Customer Location).

In [ ]:
ECHELONS = [
    {'name':'E1_Farm',         'df':e1, 'target':'Quantity_Produced',
     'keys':['Location','Product Name']},
    {'name':'E2_Distribution', 'df':e2, 'target':'Quantity_Sold',
     'keys':['Location','Product Name','Sales Channel']},
    {'name':'E3_Retail',       'df':e3, 'target':'Quantity_Sold',
     'keys':['Location','Product Name']},
]

FEATURES   = ['Year','Month','Quarter','Season','Festival_Spike_Level','Festival_Name',
              'Region_Festival_Boost','Total_Festival_Score','Is_Festival_Month','Is_High_Festival',
              'Demand_Lag_1','Demand_Lag_2','Demand_Lag_3','Demand_Lag_12',
              'Rolling_Avg_3M','Rolling_Avg_6M','Rolling_Std_3M']
print('Config set. Min length =', MIN_MONTHS, 'months.')

## Block 5 — Helper functions

In [ ]:
def mape(actual, pred):
    actual, pred = np.asarray(actual, float), np.asarray(pred, float)
    m = actual != 0
    return np.mean(np.abs((actual[m]-pred[m])/actual[m]))*100 if m.any() else np.nan

def build_series(df_grp, target):
    # Monthly target for one group; sum duplicates (E3). Keep ONLY observed months in
    # chronological order. These series are sparse over a wide span, so we do NOT reindex
    # to a continuous monthly axis (that would invent dozens of phantom months). ARIMA is
    # applied to the ordered observed sequence; every row keeps its real precomputed features.
    s = df_grp.groupby('YearMonth')[target].sum().sort_index()
    return s, s.shape[0]

def select_order(train, p_range=(0,1,2,3), d_range=(0,1), q_range=(0,1,2,3)):
    # Pick (p,d,q) by AIC on TRAIN ONLY. Returns (order, aic, used_fallback_flag).
    # used_fallback=True means no candidate converged and we fell back to (1,1,1).
    best_aic, best_order, found = np.inf, (1,1,1), False
    y = train.values.astype(float)
    for p,d,q in itertools.product(p_range, d_range, q_range):
        if p==0 and q==0:        # skip trivial
            continue
        try:
            aic = ARIMA(y, order=(p,d,q)).fit().aic
            if np.isfinite(aic) and aic < best_aic:
                best_aic, best_order, found = aic, (p,d,q), True
        except Exception:
            continue
    return best_order, (best_aic if found else np.nan), (not found)

CLIP_K = 3.0   # forecasts capped to [0, CLIP_K * max(train)] to stop ARIMA blow-ups

def clip_forecast(values, train_vals):
    # Cap forecasts to a sane demand range; return (clipped_values, n_clipped).
    hi = CLIP_K * float(np.nanmax(train_vals)) if len(train_vals) else np.inf
    lo = 0.0
    v = np.asarray(values, dtype=float)
    n_clip = int(np.sum((v < lo) | (v > hi)))
    return np.clip(v, lo, hi), n_clip

def walk_forward(train, test, order):
    # Refit fixed-order ARIMA at each step, forecast 1 ahead, then append the TRUE value.
    # Each one-step forecast is clipped to [0, CLIP_K*max(train_so_far)] to prevent drift blow-ups.
    history = list(train.values.astype(float))
    preds, n_clip = [], 0
    for t in range(len(test)):
        try:
            yhat = float(ARIMA(history, order=order).fit().forecast(steps=1)[0])
        except Exception:
            yhat = history[-1]               # naive fallback
        hi = CLIP_K * max(history)
        if not np.isfinite(yhat) or yhat < 0 or yhat > hi:
            yhat = min(max(yhat if np.isfinite(yhat) else history[-1], 0.0), hi)
            n_clip += 1
        preds.append(yhat)
        history.append(float(test.iloc[t]))  # walk forward on truth
    return np.array(preds), n_clip

print('Helpers defined.')

## Block 6 — Main loop: ARIMA over every series

For each echelon and each group that passes the length gate: fit-order on train → walk-forward forecast on test → store per-series metrics and per-month residuals. Residuals are in **original target units** (`Actual − ARIMA_Predicted`) — exactly what RF will model.

> **Forecasting protocol:** this is **one-step-ahead** walk-forward — after each prediction the true observed value is appended before predicting the next step. It measures 1-month-ahead accuracy, the standard protocol for residual-hybrid models. It is *not* a multi-month-ahead forecast. `Fallback_Order=True` flags series where no candidate order converged and `(1,1,1)` was used.

In [ ]:
series_metrics = []   # one row per series
residual_rows  = []   # rows for RF: TRAIN-period (RF trains) + TEST-period (RF evaluated)

for ech in ECHELONS:
    df, target, keys, ename = ech['df'], ech['target'], ech['keys'], ech['name']
    feat_cols = [c for c in FEATURES if c in df.columns]
    groups = df.groupby(keys)
    n_total = groups.ngroups
    n_used = n_skip = 0

    for gkey, gdf in groups:
        s, n_obs = build_series(gdf, target)
        if n_obs < MIN_MONTHS or len(s) < MIN_MONTHS:   # gate on REAL observed months
            n_skip += 1
            continue
        n_test  = max(int(round(len(s)*TEST_RATIO)), MIN_TEST)
        n_train = len(s) - n_test
        if n_train < 6:                      # need a minimum to estimate ARIMA
            n_skip += 1
            continue

        train, test = s.iloc[:n_train], s.iloc[n_train:]
        order, aic, used_fb = select_order(train)

        # In-sample (TRAIN-period) fitted values -> training residuals for RF.
        # Fit once on train; use the model's fitted values aligned to train months.
        try:
            fit_tr = ARIMA(train.values.astype(float), order=order).fit()
            train_pred = np.asarray(fit_tr.fittedvalues, dtype=float)
            # statsmodels fittedvalues has same length as train; first d values are unreliable
            if len(train_pred) != len(train):
                train_pred = np.full(len(train), np.nan)
        except Exception:
            train_pred = np.full(len(train), np.nan)
        # clip in-sample fitted values to the same sane range
        if np.isfinite(train_pred).any():
            train_pred, _ = clip_forecast(train_pred, train.values)
        train_resid = train.values - train_pred

        # Test-period one-step-ahead walk-forward (held-out evaluation), clipped
        preds, n_clip_test = walk_forward(train, test, order)   # ONE-STEP-AHEAD walk-forward
        resid = test.values - preds          # ORIGINAL units

        gkey_t = gkey if isinstance(gkey, tuple) else (gkey,)
        meta = dict(zip(keys, gkey_t))

        series_metrics.append({**meta, 'Echelon':ename, 'Order':str(order),
            'Fallback_Order':used_fb, 'N_Clipped_Test':n_clip_test,
            'n_months':len(s), 'n_train':n_train, 'n_test':n_test,
            'AIC':(round(aic,2) if np.isfinite(aic) else np.nan),
            'MAE':mean_absolute_error(test.values,preds),
            'RMSE':np.sqrt(mean_squared_error(test.values,preds)),
            'MAPE':mape(test.values,preds)})

        # per-month feature rows for the RF stage.
        # Numeric features -> mean; categorical (Season, Festival_Name) -> first.
        agg_map = {c: ('mean' if pd.api.types.is_numeric_dtype(gdf[c]) else 'first')
                   for c in feat_cols}
        fsub = gdf.groupby('YearMonth')[feat_cols].agg(agg_map)

        # TRAIN rows (RF will train on these) + TEST rows (RF evaluated on these)
        for i, dt in enumerate(train.index):
            row = {**meta, 'Echelon':ename, 'YearMonth':dt, 'Split':'train',
                   'Actual':train.values[i], 'ARIMA_Pred':train_pred[i],
                   'ARIMA_Residual':train_resid[i]}
            if dt in fsub.index:
                row.update(fsub.loc[dt].to_dict())
            residual_rows.append(row)
        for i, dt in enumerate(test.index):
            row = {**meta, 'Echelon':ename, 'YearMonth':dt, 'Split':'test',
                   'Actual':test.values[i], 'ARIMA_Pred':preds[i], 'ARIMA_Residual':resid[i]}
            if dt in fsub.index:
                row.update(fsub.loc[dt].to_dict())
            residual_rows.append(row)
        n_used += 1

    print(f'{ename:<16} groups={n_total:>4} | used={n_used:>4} | skipped(<{MIN_MONTHS}mo)={n_skip:>4}')

metrics_df  = pd.DataFrame(series_metrics)
residual_df = pd.DataFrame(residual_rows)
print('\nPer-series rows:', len(metrics_df),
      '| residual rows for RF:', len(residual_df),
      '| train:', int((residual_df['Split']=='train').sum()),
      'test:', int((residual_df['Split']=='test').sum()))

## Block 7 — Aggregate ARIMA accuracy per echelon

In [ ]:
# Median is the headline metric: short test windows make per-series MEAN MAPE explode,
# so we report MEDIAN MAPE first and keep mean as secondary context.
agg = (metrics_df.groupby('Echelon')
       .agg(n_series=('MAE','size'),
            Fallback_rate=('Fallback_Order','mean'),
            MAE_median=('MAE','median'),  MAE_mean=('MAE','mean'),
            RMSE_median=('RMSE','median'),
            MAPE_median=('MAPE','median'), MAPE_mean=('MAPE','mean'))
       .round(3))
agg['Fallback_rate'] = (agg['Fallback_rate']*100).round(1)   # as %
agg = agg.rename(columns={'Fallback_rate':'Fallback_%'})
print('ARIMA accuracy across all series  (MEDIAN MAPE is the headline metric)')
display(agg)
print(f"\nOrder-selection fallback fired on {metrics_df['Fallback_Order'].mean()*100:.1f}% of all series "
      f"({metrics_df['Fallback_Order'].sum()} of {len(metrics_df)}).")
clipped_series = (metrics_df['N_Clipped_Test'] > 0).sum()
print(f"Forecast clipping fired on {clipped_series} of {len(metrics_df)} series "
      f"({clipped_series/len(metrics_df)*100:.1f}%) — these had ARIMA forecasts capped to a sane range.")
agg.to_csv(os.path.join(RESULTS_DIR,'arima_summary_by_echelon.csv'))

In [ ]:
# Distribution of per-series MAPE per echelon (shows spread, not just the mean)
order_e = ['E1_Farm','E2_Distribution','E3_Retail']
data = [metrics_df.loc[metrics_df['Echelon']==e,'MAPE'].clip(upper=200).dropna() for e in order_e]
fig, ax = plt.subplots(figsize=(9,5))
bp = ax.boxplot(data, patch_artist=True, labels=['E1 Farm','E2 Dist','E3 Retail'],
                medianprops=dict(color='white', linewidth=2))
for patch,c in zip(bp['boxes'], PALETTE): patch.set_facecolor(c); patch.set_alpha(.75)
ax.set_ylabel('MAPE (%) per series  [clipped at 200]')
ax.set_title('ARIMA forecast error distribution by echelon', fontweight='bold')
plt.tight_layout(); plt.savefig(os.path.join(FIGURES_DIR,'arima_mape_by_echelon.png'), dpi=150)
plt.show()

## Block 8 — Example series: forecast vs actual (sanity check)

In [ ]:
# Pick the longest E2 series as a readable example
ex = metrics_df[metrics_df['Echelon']=='E2_Distribution'].sort_values('n_months', ascending=False).iloc[0]
exdf = e2[(e2['Location']==ex['Location']) & (e2['Product Name']==ex['Product Name']) &
          (e2['Sales Channel']==ex['Sales Channel'])]
s, _ = build_series(exdf, 'Quantity_Sold')
n_test = int(ex['n_test']); train, test = s.iloc[:-n_test], s.iloc[-n_test:]
order = eval(ex['Order'])
preds, _ = walk_forward(train, test, order)

fig, ax = plt.subplots(figsize=(12,5))
ax.plot(train.index, train.values, color='#AAA', label='Train')
ax.plot(test.index,  test.values,  color=PALETTE[0], lw=2, marker='o', label='Actual (test)')
ax.plot(test.index,  preds,        color=PALETTE[5], lw=2, ls='--', marker='s', label='ARIMA forecast')
ax.axvline(test.index[0], color='gray', ls=':')
ax.set_title(f"ARIMA{order} - {ex['Product Name']} | {ex['Location']} | {ex['Sales Channel']}", fontweight='bold')
ax.set_ylabel('Quantity Sold'); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(FIGURES_DIR,'arima_example_forecast.png'), dpi=150)
plt.show()

## Block 8b — ARIMA results visualizations (all series)

Parameter-selection summary (which ARIMA orders were chosen), pooled predicted-vs-actual fit, and residual distributions per echelon.

In [ ]:
# RES-1: distribution of selected ARIMA orders (parameter-selection visual)
fig, axes = plt.subplots(1,3, figsize=(16,4))
for ax,(e,c) in zip(axes, [('E1_Farm',PALETTE[0]),('E2_Distribution',PALETTE[1]),('E3_Retail',PALETTE[2])]):
    sub = metrics_df[metrics_df['Echelon']==e]
    vc = sub['Order'].value_counts().head(8)[::-1]
    ax.barh(vc.index.astype(str), vc.values, color=c, edgecolor='white')
    fb = sub['Fallback_Order'].mean()*100
    ax.set_title(f"{e} — top ARIMA orders\n(fallback {fb:.0f}%)", fontweight='bold')
    ax.set_xlabel('# series')
fig.suptitle('Selected ARIMA(p,d,q) orders by echelon (AIC on train)', fontweight='bold', y=1.05)
plt.tight_layout(); plt.savefig(os.path.join(FIGURES_DIR,'res1_order_distribution.png'), dpi=150); plt.show()

In [ ]:
# RES-2: pooled predicted vs actual across ALL test points, per echelon
fig, axes = plt.subplots(1,3, figsize=(16,5))
for ax,(e,c) in zip(axes, [('E1_Farm',PALETTE[0]),('E2_Distribution',PALETTE[1]),('E3_Retail',PALETTE[2])]):
    d = residual_df[(residual_df['Echelon']==e) & (residual_df['Split']=='test')]
    d = d.dropna(subset=['Actual','ARIMA_Pred'])
    ax.scatter(d['Actual'], d['ARIMA_Pred'], s=14, alpha=.4, color=c, edgecolors='none')
    lim = [min(d['Actual'].min(), d['ARIMA_Pred'].min()), max(d['Actual'].max(), d['ARIMA_Pred'].max())]
    ax.plot(lim, lim, 'k--', lw=1, label='perfect')
    ax.set_title(f'{e} — predicted vs actual', fontweight='bold')
    ax.set_xlabel('Actual'); ax.set_ylabel('ARIMA predicted'); ax.legend()
fig.suptitle('ARIMA fit — pooled test points by echelon', fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig(os.path.join(FIGURES_DIR,'res2_pred_vs_actual.png'), dpi=150); plt.show()

In [ ]:
# RES-3: residual distributions per echelon (these residuals are RF's target)
fig, axes = plt.subplots(1,3, figsize=(16,4))
for ax,(e,c) in zip(axes, [('E1_Farm',PALETTE[0]),('E2_Distribution',PALETTE[1]),('E3_Retail',PALETTE[2])]):
    r = residual_df.loc[(residual_df['Echelon']==e) & (residual_df['Split']=='test'),
                        'ARIMA_Residual'].dropna()
    rc = r.clip(r.quantile(.02), r.quantile(.98))   # trim extreme tails for readability
    ax.hist(rc, bins=40, color=c, edgecolor='white', alpha=.85)
    ax.axvline(0, color='black', ls='--', lw=1)
    ax.set_title(f'{e} — residuals  (mean={r.mean():.1f})', fontweight='bold')
    ax.set_xlabel('Actual − ARIMA_Pred'); ax.set_ylabel('count')
fig.suptitle('ARIMA residual distributions — the signal Random Forest will model', fontweight='bold', y=1.04)
plt.tight_layout(); plt.savefig(os.path.join(FIGURES_DIR,'res3_residual_dist.png'), dpi=150); plt.show()

## Block 9 — Save residuals + features for the Random Forest stage

`arima_residuals_for_rf.xlsx` is the bridge file. It now contains a **`Split`** column ('train' / 'test'). In Notebook 2 you will:
1. Load it; train RF **only on `Split=='train'`** rows, evaluate **only on `Split=='test'`** rows (clean separation),
2. Features X = (Month, Season, festival, lags, rolling, …), target **`ARIMA_Residual`**,
3. Hybrid forecast = `ARIMA_Pred + RF_residual_prediction`,
4. Compare ARIMA-only vs RF-only vs Hybrid per echelon on the test rows.

In [ ]:
resid_path   = os.path.join(RESULTS_DIR, 'arima_residuals_for_rf.xlsx')
metrics_path = os.path.join(RESULTS_DIR, 'arima_series_metrics.xlsx')
residual_df.to_excel(resid_path, index=False)
metrics_df.to_excel(metrics_path, index=False)
print('Saved:\n ', resid_path, f'({len(residual_df)} rows)\n ', metrics_path, f'({len(metrics_df)} series)')
print('\nResidual file columns:'); print(list(residual_df.columns))

## Block 10 — Summary & next step

- ARIMA fitted across **all qualifying series** in every echelon (length gate ≥ 10 months; skipped counts reported above).
- Non-seasonal orders chosen by **AIC on train only** — seasonality is intentionally left in the residuals for RF.
- Residuals saved in **original units** with their feature rows → `arima_residuals_for_rf.xlsx`.

**Next:** open the Random Forest notebook, load that file, predict `ARIMA_Residual`, add back to `ARIMA_Pred`, and report the ARIMA-vs-RF-vs-Hybrid comparison per echelon.